In [ ]:
using Pkg
Pkg.activate("..")

In [ ]:
include("../src/PhasorNetworks.jl")
using .PhasorNetworks, Plots, DifferentialEquations

In [ ]:
using CUDA

In [ ]:
using Statistics

In [ ]:
n_x = 101
phases = range(-1.0, 1.0, n_x) |> collect
phases = reshape(phases, (:,1));

In [ ]:
import .PhasorNetworks: spike_current, phase_memory, neuron_constant

In [ ]:
import .PhasorNetworks: bias_current, parallel_current, parallel_scatter_add

In [ ]:
spk_args = SpikingArgs()

In [ ]:
repeats = 6
tspan = (0.0, repeats*1.0)

In [ ]:
tbase = collect(tspan[1]:spk_args.solver_args[:dt]:tspan[2]);

In [ ]:
st = phase_to_train(phases, spk_args=spk_args, repeats = repeats, offset=0.0f0)

In [ ]:
sol = oscillator_bank(st, tspan=tspan, spk_args=spk_args);

In [ ]:
u = Array(sol);

In [ ]:
u |> size

In [ ]:
plot(real.(u[1,1,:]))
plot!(imag.(u[1,1,:]))

In [ ]:
scatter(real.(u[1,1,:]), imag.(u[1,1,:]))

In [ ]:
p = solution_to_phase(sol, spk_args=spk_args, final_t=false);

In [ ]:
plot(p[51,1,:])

In [ ]:
plot(p[1,:])

In [ ]:
typeof(u)

In [ ]:
st1 = solution_to_train(sol, tspan, spk_args = spk_args, offset=0.0)

In [ ]:
p1 = train_to_phase(st1, spk_args=spk_args)

In [ ]:
err2 = p1[end-1,:,1] .- phases;

In [ ]:
scatter(p1[end-1,:,1], phases)

In [ ]:
histogram(arc_error(err2))

In [ ]:
stg = SpikeTrainGPU(st)

In [ ]:
using ChainRulesCore: ignore_derivatives

In [ ]:
import .PhasorNetworks: gaussian_kernel

In [ ]:
stg.indices

In [ ]:
stg.linear_indices

In [ ]:
stg.shape

In [ ]:
stg.linear_shape

In [ ]:
# Set up sample data
n = 1_00
m = 100_000

# Create indices and values on the GPU
d_indices = rand(1:n, m) |> CuArray{Int}
d_values = CUDA.rand(Float32, m);

In [ ]:
typeof(m)

In [ ]:
d_indices

In [ ]:
v = parallel_scatter_add(d_indices, d_values, n)

In [ ]:
for i in 1:1000 parallel_scatter_add(d_indices, d_values, n) end

In [ ]:
for i in 1:10000 parallel_scatter_add(d_indices, d_values, n) end

In [ ]:
#for i in 1:100000 parallel_scatter_add(d_indices, d_values, n) end

In [ ]:
v

In [ ]:
typeof(spk_args.t_window)

In [ ]:
import .PhasorNetworks: gaussian_kernel_gpu

In [ ]:
i0 = gaussian_kernel_gpu.(stg.times, 0.4f0, Float32(spk_args.t_window))

In [ ]:
for i in 1:1000 gaussian_kernel_gpu.(stg.times, 0.4f0, Float32(spk_args.t_window)) end

In [ ]:
for i in 1:10000 gaussian_kernel_gpu.(stg.times, 0.4f0, Float32(spk_args.t_window)) end

In [ ]:
ik = parallel_current(stg, 0.1f0, spk_args)

In [ ]:
iv = stack([parallel_current(stg, t, spk_args) for t in 0.0f0:0.01f0:1.1f0])

In [ ]:
iv2 = Array(iv);

In [ ]:
heatmap(iv2)

In [ ]:
ival = spike_current(stg, 0.4f0, spk_args)

In [ ]:
plot(Array(ival))

In [ ]:
@time ivals = stack([Array(spike_current(stg, t, spk_args)) for t in 0.0f0:0.001f0:1.0f0], dims=1)

In [ ]:
heatmap(ivals)

In [ ]:
@time ivals2 = stack([Array(spike_current(st, t, spk_args)) for t in 0.0:0.001:1.0], dims=1);

In [ ]:
sum(abs2.(ivals .- ivals2))

In [ ]:
heatmap(ivals2)

In [ ]:
spk_args

In [ ]:
p1 = oscillator_bank(stg, spk_args=spk_args)

In [ ]:
ph1 = solution_to_phase(p1, spk_args=spk_args)

In [ ]:
ph1c = Array(ph1)

In [ ]:
plot(ph1c[:,1,:]')

In [ ]:
scatter(ph1c[:,1,end-1], phases)

In [ ]:
biases = angle_to_complex(range(start=-1.0f0, stop=1.0f0, length=51)) |> cu

In [ ]:
typeof(biases) <: CuArray{<:Complex}

In [ ]:
bias_current(biases, 0.4f0, 0.0f0, spk_args)

In [ ]:
bivals = stack([bias_current(biases, t, 0.0f0, spk_args) for t in 0.0f0:0.01f0:1.0f0])

In [ ]:
heatmap(Array(bivals))

In [ ]:
update_fn = spk_args.update_fn

In [ ]:
w = CUDA.ones((51,101));

In [ ]:
w * bivals[1,:]

In [ ]:
du = CUDA.zeros(ComplexF32, (51,))
u = CUDA.zeros(ComplexF32, (51,));

In [ ]:
x = stg

In [ ]:
b = biases;

In [ ]:
tspan[2]

In [ ]:
sol2 = oscillator_bank(stg, w, b, tspan=tspan, spk_args=spk_args)